# 16 · Vision – the Avengers assemble 🧪

Vision is not one mind: it was built from J.A.R.V.I.S., the Mind Stone and Tony's and Bruce's work. Our finale connects **independent agents** into one system with **A2A** (*Agent2Agent*), the open protocol for agents to discover and call each other.

```
you ─▶ jarvis-orchestrator ──(A2A tool)──▶ friday-finance  (own endpoint, own instructions)
              └──(MCP tool)──▶ stark-mcp (workshop facts)
```

| 🧱 Part | 🔎 What it is |
|---|---|
| **Agent card** | the discovery document of an A2A agent (description, skills) |
| **Incoming A2A** | switch on the A2A endpoint of a Foundry prompt agent (`update_details`) |
| **A2A connection** | project connection that stores the remote agent's URL and auth |
| **A2A tool** | `A2ATool` on the calling agent |

> 🧪 **Preview-adjacent.** A2A v1.0 is GA in Foundry; the Python SDK surface (`update_details`, `A2ATool`) needs `azure-ai-projects>=2.5.0`. Limits: text only, no streaming, JSON-RPC only. Calls need Entra ID and the **Foundry Agent Consumer** role. Deep dive: [docs/mcp-a2a-skills.md](../docs/mcp-a2a-skills.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

import json, requests
from azure.ai.projects.models import (A2AProtocolConfiguration, A2AProtocolVersion, A2ATool, AgentCard, AgentCardSkill,
                                      AgentEndpointConfig, MCPTool, ProtocolConfiguration, PromptAgentDefinition,
                                      ResponsesProtocolConfiguration)
oai = openai_client()
created = []

## 1 · The remote specialist: `friday-finance`
A normal prompt agent. Its instructions embed the **invoice-review skill** (direct injection, chapter 07) and it can use our MCP server's `check_invoice_approval` tool.

In [ ]:
friday = project().agents.create_version(
    agent_name="friday-finance",
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions="You are F.R.I.D.A.Y., the finance specialist. Review invoices strictly with this checklist:\n" + skill_body("invoice-review"),
        tools=[MCPTool(server_label="stark", server_url=cfg("STARK_MCP_URL"), project_connection_id="stark-mcp",
                       require_approval="never", allowed_tools=["check_invoice_approval"])],
    ),
)
created.append("friday-finance")
print("created", friday.name, "v", friday.version)

## 2 · Expose it over A2A
One call sets the **agent card** and turns on the A2A protocol next to Responses.

In [ ]:
project().agents.update_details(
    agent_name="friday-finance",
    agent_endpoint=AgentEndpointConfig(protocol_configuration=ProtocolConfiguration(
        responses=ResponsesProtocolConfiguration(), a2a=A2AProtocolConfiguration())),
    agent_card=AgentCard(version="1.0", description="Reviews supplier invoices and checks approval limits.",
        skills=[AgentCardSkill(id="invoice-review", name="Invoice review", description="Reviews invoice data against the checklist and approval limits.")]),
)
A2A_BASE = f"{cfg('STARK_PROJECT_ENDPOINT')}/agents/friday-finance/endpoint/protocols/a2a"
print(A2A_BASE)

## 3 · Discover it like any A2A client
Plain HTTP with an Entra token (scope `https://ai.azure.com/.default`). Anonymous access is not supported. Version **v1.0** card:

In [ ]:
token = credential().get_token("https://ai.azure.com/.default").token
card = requests.get(f"{A2A_BASE}/agentCard/v1.0", headers={"Authorization": f"Bearer {token}"}).json()
print(json.dumps({k: card.get(k) for k in ("name", "description", "protocolVersion", "skills")}, indent=2)[:900])

### Call it with the open-source A2A SDK
The `a2a-sdk` reads `protocolVersion` from the card and negotiates v1.0. This is exactly what a non-Foundry agent (LangGraph, your own service) would do.

In [ ]:
import httpx
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import new_text_message
from a2a.types.a2a_pb2 import Role, SendMessageRequest

async with httpx.AsyncClient(headers={"Authorization": f"Bearer {token}"}, timeout=httpx.Timeout(120.0)) as http:
    agent_card = await A2ACardResolver(httpx_client=http, base_url=A2A_BASE, agent_card_path="agentCard/v1.0").get_agent_card()
    a2a = await create_client(agent=agent_card, client_config=ClientConfig(streaming=False, httpx_client=http))
    request = SendMessageRequest(message=new_text_message("Invoice 9930 from Oscorp Materials totals 99,792 USD. Can Pepper Potts approve it?", role=Role.ROLE_USER))
    async for response in a2a.send_message(request):
        print(response)
    await a2a.close()

## 4 · Let another Foundry agent call it
Step 1: a **connection** to the target (ARM). Step 2: an agent with the **A2A tool**. The caller's identity (agent identity) needs the *Foundry Agent Consumer* role on the target.

In [ ]:
PROJECT_PATH = f"{account_path()}/projects/{cfg('STARK_FOUNDRY_PROJECT')}"
CONNECTION_PATH = f"{PROJECT_PATH}/connections/friday-a2a"
CONNECTION_API = "2025-04-01-preview"
body = {"properties": {"authType": "AgenticIdentityToken", "category": "RemoteA2A", "target": A2A_BASE,
                       "audience": "https://ai.azure.com", "Credentials": {}, "metadata": {}}}
r = arm("PUT", CONNECTION_PATH, CONNECTION_API, body)
print(r.status_code, "" if r.ok else r.text[:300])

In [ ]:
conn = project().connections.get("friday-a2a")
orchestrator = project().agents.create_version(
    agent_name="jarvis-orchestrator",
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions=("You are J.A.R.V.I.S., the orchestrator. For anything about invoices or approvals, delegate to F.R.I.D.A.Y. "
                      "with the A2A tool and relay her answer. For suits and workshop facts use the stark tools. Be brief."),
        tools=[A2ATool(a2a_version=A2AProtocolVersion.V1_0, project_connection_id=conn.id),
               MCPTool(server_label="stark", server_url=cfg("STARK_MCP_URL"), project_connection_id="stark-mcp",
                       require_approval="never", allowed_tools=["list_suits", "get_suit", "get_workshop_status"])],
    ),
)
created.append("jarvis-orchestrator")
print("created", orchestrator.name, "v", orchestrator.version)

In [ ]:
# Grant the orchestrator's own identity access to the target: the Foundry Agent Consumer role on the project.
import uuid
AGENT_CONSUMER = "eed3b665-ab3a-47b6-8f48-c9382fb1dad6"   # built-in role definition id
try:
    principal = project().agents.get(agent_name="jarvis-orchestrator").instance_identity.principal_id
    role = f"/subscriptions/{cfg('AZURE_SUBSCRIPTION_ID')}/providers/Microsoft.Authorization/roleDefinitions/{AGENT_CONSUMER}"
    name = uuid.uuid5(uuid.NAMESPACE_URL, PROJECT_PATH + principal + AGENT_CONSUMER)   # same name on a re-run, no duplicates
    ra = arm("PUT", f"{PROJECT_PATH}/providers/Microsoft.Authorization/roleAssignments/{name}", "2022-04-01",
             {"properties": {"principalId": principal, "principalType": "ServicePrincipal", "roleDefinitionId": role}})
    print("role assignment:", ra.status_code, "(already assigned)" if ra.status_code == 409 else ("" if ra.ok else ra.text[:200]))
except Exception as exc:
    print("Assign 'Foundry Agent Consumer' to the orchestrator's identity manually:", type(exc).__name__)

## 5 · The whole system in one question
Role assignments need a minute to propagate; retry if you see a 403.

In [ ]:
import time
conv = oai.conversations.create()
for attempt in range(3):
    try:
        r = ask_agent("jarvis-orchestrator", "Is the arc reactor online, and can Pepper Potts approve invoice 9930 for 99,792 USD?", conversation=conv.id)
        print(r.output_text)
        print("\ntool trail:", tool_calls(r))
        break
    except Exception as exc:
        print("retry", attempt + 1, type(exc).__name__, str(exc)[:120]); time.sleep(45)

Open **Traces** (chapter 03) and the conversation: you see the orchestrator's A2A call and, as a separate agent run, Friday's own work with her MCP call – two agents, two identities, one trace story.

## 6 · Prove it (chapter 10 again)
Run the quality gate on the orchestrator before you trust it:

In [ ]:
queries = project().datasets.upload_file(name="stark-capstone", version="1", file_path=str(ROOT / "data" / "capstone-queries.jsonl"))
ev = oai.evals.create(name="capstone", data_source_config=item_schema({"query": {"type": "string"}}, required=["query"]),
    testing_criteria=[criterion("task_adherence", "task_adherence", {"query": "{{item.query}}", "response": "{{sample.output_items}}"})])
run = oai.evals.runs.create(eval_id=ev.id, name="orchestrator", data_source=agent_target(queries.id, "jarvis-orchestrator"))
run = wait_eval_run(ev.id, run.id, oai)
try:
    assert_pass_rate(run, minimum=0.6)
except AssertionError as exc:
    print("GATE FAILED:", exc)

## 👀 In the portal
**Operate → Assets** 🧪 and **Build → Agents**: `jarvis-orchestrator` and `friday-finance`; **Evaluation**: `capstone`. Compare their traces.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
oai.conversations.delete(conv.id)
try: oai.evals.delete(ev.id)
except Exception: pass
for name in created:
    delete_agent(name)
print(arm("DELETE", CONNECTION_PATH, CONNECTION_API).status_code, "(connection friday-a2a)")
delete_dataset("stark-capstone")

Remove the **whole workshop** from Azure (models, monitoring, function): `azd down --purge`.

## 🎯 Recap: the moral
You started with a robot arm that could answer one question. Now you have models, memory, tools, skills, code-hosted agents, evidence, senses, guardrails, workflows and agents that talk to each other. **Foundry is the workshop; you decide what to build.**